# 12 — Nonparametric Inference
**References:** Wasserman (2004) *All of Nonparametric Statistics* Ch. 1–4 · Lehmann (1975) · Efron & Hastie (2016) Ch. 10–11

## Narrative thread
```
Empirical CDF -> Bootstrap theory -> Rank tests -> Permutation tests -> KDE
```

## 1. The empirical distribution function

Given i.i.d. $X_1,\ldots,X_n$ from $F$, the **empirical CDF** is:
$$\hat{F}_n(x) = \frac{1}{n}\sum_{i=1}^n \mathbf{1}(X_i \leq x)$$

**Glivenko-Cantelli theorem:** $\sup_x |\hat{F}_n(x) - F(x)| \xrightarrow{a.s.} 0$

**Dvoretzky-Kiefer-Wolfowitz (DKW) inequality:**
$$P\!\left(\sup_x |\hat{F}_n(x) - F(x)| > \varepsilon\right) \leq 2e^{-2n\varepsilon^2}$$

This gives a **simultaneous confidence band** for $F$:
$$P\!\left(\hat{F}_n(x) - \varepsilon_n \leq F(x) \leq \hat{F}_n(x) + \varepsilon_n \;\forall\, x\right) \geq 1-\alpha$$
where $\varepsilon_n = \sqrt{\log(2/\alpha)/(2n)}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── ECDF with DKW confidence band ─────────────────────────────────────────
n     = 50
alpha = 0.05
data  = np.random.gamma(3, 2, n)
true_dist = stats.gamma(a=3, scale=2)

x_sorted = np.sort(data)
ecdf     = np.arange(1, n+1) / n
eps_n    = np.sqrt(np.log(2/alpha) / (2*n))

x_plot = np.linspace(0, 20, 400)

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(x_sorted, ecdf, color='#4361ee', lw=2, where='post', label='ECDF $\\hat{F}_n$')
ax.fill_between(x_sorted, np.clip(ecdf - eps_n, 0, 1), np.clip(ecdf + eps_n, 0, 1),
                step='post', alpha=0.2, color='#4361ee', label=f'DKW 95% band ($\\varepsilon_n={eps_n:.3f}$)')
ax.plot(x_plot, true_dist.cdf(x_plot), 'r-', lw=2.5, label='True $F$ — Gamma(3,2)')
ax.set_xlabel('x'); ax.set_ylabel('$F(x)$')
ax.set_title(f'ECDF with DKW simultaneous confidence band (n={n}, α={alpha})\n'
              'Band covers the true CDF everywhere with probability ≥ 95%')
ax.legend()
plt.tight_layout()
plt.show()

## 2. The Bootstrap

**Efron's bootstrap (1979):** Estimate the sampling distribution of $T_n = g(X_1,\ldots,X_n)$ by resampling.

**Algorithm:**
1. Draw $B$ bootstrap samples $\mathbf{X}^{*(b)} = (X^*_1,\ldots,X^*_n)$ with replacement from $\{X_1,\ldots,X_n\}$
2. Compute $T_n^{*(b)} = g(X^{*(b)}_1,\ldots,X^{*(b)}_n)$ for each $b$
3. Use the distribution of $\{T_n^{*(b)}\}$ to estimate the sampling distribution of $T_n$

**Bootstrap SE:** $\widehat{\text{SE}}_{\text{boot}} = \sqrt{\frac{1}{B-1}\sum_b (T_n^{*(b)} - \bar T_n^*)^2}$

**Bootstrap CI methods:**

| Method | Formula | Notes |
|---|---|---|
| Normal | $\hat\theta \pm z_{\alpha/2}\cdot\widehat{\text{SE}}_{\text{boot}}$ | Assumes symmetry |
| Percentile | $[Q^*_{\alpha/2},\, Q^*_{1-\alpha/2}]$ | Simple but biased |
| Pivotal (basic) | $[2\hat\theta - Q^*_{1-\alpha/2},\, 2\hat\theta - Q^*_{\alpha/2}]$ | Corrects bias |
| BCa | Bias-corrected + accelerated | Best coverage; complex |

In [ ]:
# ── Bootstrap CI for the median (no closed-form SE) ───────────────────────
np.random.seed(42)
n_data = 40
data_b = np.random.exponential(scale=3, size=n_data)
true_median = np.log(2) * 3  # median of Exp(1/3) is ln(2)*3

B = 5000
boot_medians = [np.median(np.random.choice(data_b, n_data, replace=True)) for _ in range(B)]
boot_medians = np.array(boot_medians)

theta_hat = np.median(data_b)
boot_se   = boot_medians.std()

# Three CI types
ci_normal   = (theta_hat - 1.96*boot_se, theta_hat + 1.96*boot_se)
ci_pct      = np.percentile(boot_medians, [2.5, 97.5])
ci_pivotal  = (2*theta_hat - ci_pct[1], 2*theta_hat - ci_pct[0])

print(f'True median of Exp(3): {true_median:.4f}')
print(f'Sample median: {theta_hat:.4f}')
print(f'Bootstrap SE:  {boot_se:.4f}')
print()
print(f'Normal CI:    [{ci_normal[0]:.4f}, {ci_normal[1]:.4f}]  contains true: {ci_normal[0]<=true_median<=ci_normal[1]}')
print(f'Percentile CI:[{ci_pct[0]:.4f}, {ci_pct[1]:.4f}]  contains true: {ci_pct[0]<=true_median<=ci_pct[1]}')
print(f'Pivotal CI:   [{ci_pivotal[0]:.4f}, {ci_pivotal[1]:.4f}]  contains true: {ci_pivotal[0]<=true_median<=ci_pivotal[1]}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(boot_medians, bins=60, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
axes[0].axvline(theta_hat,   color='#f72585', lw=2, linestyle='--', label=f'$\\hat\\theta={theta_hat:.3f}$')
axes[0].axvline(true_median, color='#06d6a0', lw=2, linestyle=':',  label=f'True med={true_median:.3f}')
for lo, hi, name in [(ci_pct[0], ci_pct[1], 'Pct 95% CI')]:
    axes[0].axvspan(lo, hi, alpha=0.15, color='#f72585', label=name)
axes[0].set_title(f'Bootstrap distribution of sample median (B={B})\nExp(3) data, n={n_data}')
axes[0].legend(fontsize=8)

# Coverage comparison via simulation
n_sim   = 2000
B_sim   = 1000
covers  = {'Normal': 0, 'Percentile': 0, 'Pivotal': 0}

for _ in range(n_sim):
    d = np.random.exponential(3, n_data)
    th = np.median(d)
    bm = np.array([np.median(np.random.choice(d, n_data, replace=True)) for _ in range(B_sim)])
    se = bm.std()
    q  = np.percentile(bm, [2.5, 97.5])
    covers['Normal']     += (th - 1.96*se <= true_median <= th + 1.96*se)
    covers['Percentile'] += (q[0] <= true_median <= q[1])
    covers['Pivotal']    += (2*th - q[1] <= true_median <= 2*th - q[0])

names, covs = zip(*[(k, v/n_sim) for k,v in covers.items()])
bars = axes[1].bar(names, covs, color=['#4361ee','#f72585','#06d6a0'], alpha=0.8, width=0.4)
axes[1].axhline(0.95, color='k', lw=1.5, linestyle='--', label='Nominal 95%')
for b, c in zip(bars, covs):
    axes[1].text(b.get_x()+b.get_width()/2, c+0.003, f'{c:.3f}', ha='center', fontweight='bold')
axes[1].set_ylabel('Empirical coverage'); axes[1].set_ylim(0.85, 1.0)
axes[1].set_title(f'Bootstrap CI coverage (n_sim={n_sim}, B={B_sim})\nPivotal usually closest to nominal 95%')
axes[1].legend()

plt.suptitle('Bootstrap Confidence Intervals — Coverage Comparison', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Rank-based tests

**Wilcoxon signed-rank test:** For paired differences $D_i = X_i - Y_i$, tests $H_0: \text{median}(D)=0$.

$$W^+ = \sum_{i: D_i > 0} \text{rank}(|D_i|)$$

Under $H_0$, $E[W^+] = n(n+1)/4$ and $\text{Var}(W^+) = n(n+1)(2n+1)/24$.

**Wilcoxon rank-sum (Mann-Whitney):** For two independent samples, tests $H_0: F_X = F_Y$.

**ARE of rank tests vs t-tests:**

| Parent distribution | ARE(Wilcoxon, t) |
|---|---|
| Normal | $3/\pi \approx 0.955$ — nearly efficient |
| Uniform | $1.0$ |
| Laplace | $1.5$ — 50% more powerful |
| Cauchy | $\infty$ — t-test breaks down |

> Rank tests never lose much vs t-test under normality, but can be much better under heavy tails.

## 4. Permutation tests

**Permutation test:** Under $H_0$ that the groups are exchangeable, the observed test statistic $T_\text{obs}$ should be no more extreme than a random permutation of group labels.

$$p\text{-value} = \frac{\#\{T^{(b)} \geq T_\text{obs}\}}{B}$$

**Key property:** The permutation p-value is **exact** — no asymptotic approximation needed.

| Feature | Parametric | Bootstrap | Permutation |
|---|---|---|---|
| Requires distribution assumption | Yes | No | No |
| Valid for small n | Depends | Approximate | Yes (exact) |
| Computes | CI + p-value | CI + SE | p-value only |
| Null distribution | Theoretical | Resampling | Permutations |